In [16]:
import duckdb
import pandas as pd
import numpy as np
import csv

import matplotlib.pyplot as plt # install in POWERSHELL using "python -m pip install matplotlib.pyplot"
import seaborn as sns

import torch

%matplotlib inline

plt.style.use("ggplot")

con = duckdb.connect("main.duckdb")

In [8]:
# Only run at the end to save.
con.close()

In [3]:
# Set up, you only need to follow the instructions and run the first time!
# 1. Download the dataset from the census FTP (https://www2.census.gov/programs-surveys/acs/data/pums/2022/5-Year/csv_pus.zip) or try https://www2.census.gov/programs-surveys/acs/data/pums/2022/5-Year/ and navigate to csv_pus.zip
# 2. Unzip the file (may take ~10 minutes for slow computers). Make sure the folder is named "csv_pus" and the files follow the pattern "psam_pusa.csv", "psam_pusb.csv"...
# 3 Place the unzipped folder into the directory
# 4 Run this cell, which make take around 30 minutes or up to an hour on slower computers. Try to keep the computer on and just watch netflix or something on the side :)

  # Not working? You probably don't have python or duckdb installed. Use powershell (or bash if you really want) to verify that python is installed on your machine and then pip install duckdb and pandas. Ask Noam about it or use this link to try and solve it yourself (https://opentechschool.github.io/python-beginners/en/getting_started.html#what-is-python-exactly)

# Combine all the tables into table called "main" (takes ~30mins)
con.sql("""
  CREATE OR REPLACE TABLE main AS
    SELECT * FROM read_csv('csv_pus/psam_pusa.csv')
    UNION
    SELECT * FROM read_csv('csv_pus/psam_pusb.csv')
    UNION  
    SELECT * FROM read_csv('csv_pus/psam_pusc.csv')
    UNION  
    SELECT * FROM read_csv('csv_pus/psam_pusd.csv')
""")

ConnectionException: Connection Error: Connection already closed!

In [9]:
# Function for comparing table sizes and storing it in memory. Useful for seeing if your filter reduced the table to a reasonable size. 
entry_memory = {}

def compare_tables(table1, table2):
    res = con.execute(f"""
    SELECT 
        (SELECT COUNT(*) FROM "{table1}") as count1, 
        (SELECT COUNT(*) FROM "{table2}") as count2
    """).fetchone()
    c1, c2 = res[0], res[1]
    
    entry_memory[table1], entry_memory[table2] = c1, c2
    print(f"{table1}: {c1:,} | {table2}: {c2:,} | Diff: {(c2 - c1):,} ({'more' if c2 - c1 >= 0 else 'less'})")

In [ ]:
# con.sql("""DESCRIBE main """).show(max_rows=999)

# con.sql("""SELECT * FROM main""")

┌─────────┬───────────────┬──────────┬─────────┬─────────┬────────┬────────┬─────────┬─────────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬─────────┬───────┬───────┬───────┬───────┬───────┬───────┬────────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬──────────┬───────┬───────┬─────────┬─────────┬───────┬───────┬───────┬───────┬────────┬───────┬───────┬───────┬─────────┬───────┬───────┬───────┬─────────┬─────────┬────────┬───────┬─────────┬───────┬───────┬───────┬───────┬───────┬─────────┬─────────┬─────────┬─────────┬───────┬───────────┬───────────┬─────────┬───────┬─────────┬──────────┬───────┬───────┬─────────┬───────┬────────┬────────┬─────────┬────────┬───────────┬───────────┬─────────┬─────────┬────────┬────────┬

Filter

See [data dictionary documentation](https://www2.census.gov/programs-surveys/acs/tech_docs/pums/data_dict/PUMS_Data_Dictionary_2022.pdf) for details

In [10]:
# Get the data dictonary setup so you can easily translate SQL output
with open('PUMS_Data_Dictionary_2018-2022.csv', newline='') as csvfile:
    reader = csv.reader(csvfile, delimiter=",")
    output = list(reader)

# Create data_dict as a resource, then step through the document
data_dict = {}
for row in output:
    # column 1 is always NAME or VAL
    if row[0] == "NAME": 
        # IF NAME, then the second column is the name of the var, the third is the type, and the fifth is a description
        data_dict[row[1]] = {"Desc": row[4], "type": row[2],  "values": {}}
    if row[0] == "VAL":
        # IF VALUE check for type (EITHER N: Numeric, C: Character)
        if row[2] == "N": # if Numeric
            try: 
                value = int(row[5])
            except (ValueError, TypeError):
                value = str(row[5])
            if row[4] == row[5]: # if the min and the max are the same (e.g. range 1 to 1)
                data_dict[row[1]]["values"][value] = row[6]
            # Don't collect data dictionary for anything that has a range that isn't the same. An integer doesn't need a translation.

        else: # if Character
            value = str(row[5])
            data_dict[row[1]]["values"][value] = row[6]


# Function to use in order to translate dataframes or certain columns using the data-dict. (NOT FULLY TESTED!!!)
def translate(target_df, target_column = "*", translate_dict = data_dict):
    return_df = target_df.copy() # copy to avoid breaking anything
    if target_column == "*": # ALL
        xtarget = return_df
    elif isinstance(target_column, str): # string, single column, needs to be put into a list
        target_column = [target_column]
        xtarget = return_df[target_column]
    else: # handle lists
        xtarget = return_df[target_column]

    # loop through target and translate
    for series_name, series in xtarget.items():
        if series_name in translate_dict.keys():
            if translate_dict[series_name]["type"] == "C":
                return_df = return_df.astype({series_name: "str"})            
            
            return_df[series_name] = return_df[series_name].replace(translate_dict[series_name]["values"])
    
    return return_df

# Usage Guide:
    # df: plug in your whole dataframe. NO CHANGES WILL BE MADE TO THIS DATAFRAME.
    # target_column (default = ALL [*]): takes list of column names, single column name, or * (for all) and uses as target for transformation
    # translate_dict (default = data_dict): replacement translation, only for advanced purposes, defaults will work for all required work
    # Returns updated dataframe. 


In [13]:
# GEOGRAPHIC & OTHER FILTER LAYER

# Apply region, division, or state filter as well as any other column filters which go into a new table called "main_refined"
alt = {"ESR": "1"} 

# ESR - Employment Status = Employed

query = f"""
    SELECT *,
    CASE 
      WHEN JWTRNS IN ('01', '08') THEN 'Private Transit' -- "Car, truck, or van", "Motorcycle"
      WHEN JWTRNS IN ('02', '03', '04', '05', '06', '07') THEN 'Public Transit' -- "Bus", "Subway or elevated rail", "Long-distance train or commuter rail", "Light rail, streetcar, or trolley", "Ferryboat", "Taxicab"
      WHEN JWTRNS IN ('09', '10') THEN 'Soft Transit' -- "Bicycle", "Walked"
      ELSE 'Other' -- "Worked from home", "Other mode", NULLs (if they exist)
    END AS 'TRANSIT'
    FROM main
    WHERE 1=1"""


for key, value in alt.items():
  query += f""" AND {key} = {value!r}"""

print(query)

con.sql("""CREATE OR REPLACE TABLE main_refined AS""" + query)

compare_tables("main", "main_refined")

##### REGION CODE ("REGION")

# 1 Northeast
# 2 Midwest
# 3 South
# 4 West
# 9 Puerto Rico

##### DIVISION CODE ("DIVISION")
# 0 Puerto Rico                              # 1 New England (Northeast region)
# 2 Middle Atlantic (Northeast region)       # 3 East North Central (Midwest region)
# 4 West North Central (Midwest region)      # 5 South Atlantic (South region)
# 6 East South Central (South region)        # 7 West South Central (South Region)
# 8 Mountain (West region)                   # 9 Pacific (West region)


##### STATE CODE ("ST")
# 01 Alabama/AL                 # 21 Kentucky/KY            # 38 North Dakota/ND
# 02 Alaska/AK                  # 22 Louisiana/LA           # 39 Ohio/OH
# 04 Arizona/AZ                 # 23 Maine/ME               # 40 Oklahoma/OK
# 05 Arkansas/AR                # 24 Maryland/MD            # 41 Oregon/OR
# 06 California/CA              # 25 Massachusetts/MA       # 42 Pennsylvania/PA
# 08 Colorado/CO                # 26 Michigan/MI            # 44 Rhode Island/RI
# 09 Connecticut/CT             # 27 Minnesota/MN           # 45 South Carolina/SC
# 10 Delaware/DE                # 28 Mississippi/MS         # 46 South Dakota/SD
# 11 District of Columbia/DC    # 29 Missouri/MO            # 47 Tennessee/TN
# 12 Florida/FL                 # 30 Montana/MT             # 48 Texas/TX
# 13 Georgia/GA                 # 31 Nebraska/NE            # 49 Utah/UT
# 15 Hawaii/HI                  # 32 Nevada/NV              # 50 Vermont/VT
# 16 Idaho/ID                   # 33 New Hampshire/NH       # 51 Virginia/VA
# 17 Illinois/IL                # 34 New Jersey/NJ          # 53 Washington/WA
# 18 Indiana/IN                 # 35 New Mexico/NM          # 54 West Virginia/WV
# 19 Iowa/IA                    # 36 New York/NY            # 55 Wisconsin/WI
# 20 Kansas/KS                  # 37 North Carolina/NC      # 56 Wyoming/WY
#                                                           # 72 Puerto Rico/PR

# Map view of codes: https://www2.census.gov/geo/pdfs/maps-data/maps/reference/us_regdiv.pdf


    SELECT *,
    CASE 
      WHEN JWTRNS IN ('01', '08') THEN 'Private Transit' -- "Car, truck, or van", "Motorcycle"
      WHEN JWTRNS IN ('02', '03', '04', '05', '06', '07') THEN 'Public Transit' -- "Bus", "Subway or elevated rail", "Long-distance train or commuter rail", "Light rail, streetcar, or trolley", "Ferryboat", "Taxicab"
      WHEN JWTRNS IN ('09', '10') THEN 'Soft Transit' -- "Bicycle", "Walked"
      ELSE 'Other' -- "Worked from home", "Other mode", NULLs (if they exist)
    END AS 'TRANSIT'
    FROM main
    WHERE 1=1 AND ESR = '1'
main: 15,721,123 | main_refined: 7,081,581 | Diff: -8,639,542 (less)


--- 
### --- Analysis Below --- 
--- 

In [14]:
con.sql("""
    CREATE OR REPLACE TABLE main_analysis AS
        (SELECT
        ST, DIVISION, REGION,
        AGEP, SEX, MAR,
        NATIVITY, ENG, RAC1P,
        DIS, DPHY, DEAR, DEYE,
        PINCP, PAP,
        COW, WKHP, 
        TRANSIT,
        SUM(PWGTP) AS 'PERSONS'
        FROM main_refined
        GROUP BY ALL)
""")

In [15]:
compare_tables("main_refined", "main_analysis")

main_refined: 7,081,581 | main_analysis: 6,242,052 | Diff: -839,529 (less)


In [ ]:
# Models Go Here!